# Chapter 13: Interfaces, Protocols, and ABCs
*Fluent Python, 2nd Edition -- Luciano Ramalho*

Python offers **four complementary approaches** to typing:
| Approach | Mechanism | Checked |
|---|---|---|
| Duck typing | Informal protocols | Runtime (interpreter) |
| Goose typing | ABCs + isinstance | Runtime (explicit) |
| Static typing | Nominal (PEP 484) | Static (mypy) |
| Static duck typing | typing.Protocol (PEP 544) | Static + optional runtime |

This notebook explores all four through runnable examples.

---
## 1. Dynamic Protocols and Duck Typing

Python's interpreter goes out of its way to cooperate with objects that provide
even a **partial** protocol implementation. Implementing just `__getitem__` is
enough to get iteration, containment checks, and indexing.

In [ ]:
class Vowels:
    """Partial sequence protocol -- only __getitem__."""
    def __getitem__(self, i):
        return 'AEIOU'[i]

v = Vowels()
print("Index:  ", v[0], v[-1])
print("Iter:   ", [c for c in v])
print("In:     ", 'E' in v, '|', 'Z' in v)

### The FrenchDeck -- a full sequence protocol

Implementing both `__getitem__` and `__len__` gives you a complete
**immutable sequence**: indexing, slicing, `len()`, iteration, `in`, and
even `random.choice`.

In [ ]:
import collections
import random

Card = collections.namedtuple('Card', ['rank', 'suit'])

class FrenchDeck:
    ranks = [str(n) for n in range(2, 11)] + list('JQKA')
    suits = 'spades diamonds clubs hearts'.split()

    def __init__(self):
        self._cards = [Card(rank, suit) for suit in self.suits
                       for rank in self.ranks]

    def __len__(self):
        return len(self._cards)

    def __getitem__(self, position):
        return self._cards[position]

deck = FrenchDeck()
print(f"Cards in deck: {len(deck)}")
print(f"First card:    {deck[0]}")
print(f"Random pick:   {random.choice(deck)}")
print(f"Top 3:         {deck[:3]}")

### Monkey Patching: Adding Protocol Methods at Runtime

`random.shuffle` requires a **mutable** sequence (`__setitem__`).
We can add it *at runtime* via monkey patching.

In [ ]:
# FrenchDeck from above lacks __setitem__, so shuffle fails:
try:
    random.shuffle(deck)
except TypeError as e:
    print(f"Expected error: {e}")

# Monkey-patch __setitem__ onto the class
def set_card(deck, position, card):
    deck._cards[position] = card

FrenchDeck.__setitem__ = set_card

# Now shuffle works!
random.shuffle(deck)
print(f"After shuffle, first 5: {deck[:5]}")

### Defensive Programming and Fail Fast

Instead of type-checking arguments, **convert eagerly** and let
Python raise clear errors. This is the EAFP style (Easier to Ask
Forgiveness than Permission).

In [ ]:
def safe_init(iterable):
    """Defensive: convert to list immediately -- fail fast."""
    items = list(iterable)   # TypeError if not iterable
    return items

# Works with any iterable
print(safe_init(range(5)))
print(safe_init("abc"))
print(safe_init({10, 20, 30}))

# Fails fast with a clear error
try:
    safe_init(42)
except TypeError as e:
    print(f"Fail fast: {e}")

#### Duck-typing to handle str-or-iterable (namedtuple pattern)

In [ ]:
def parse_field_names(field_names):
    """Accept a comma/space-separated string OR an iterable of strings."""
    try:
        # Assume it is a string (EAFP)
        field_names = field_names.replace(',', ' ').split()
    except AttributeError:
        pass  # Not a string -- assume iterable of strings
    field_names = tuple(field_names)
    if not all(s.isidentifier() for s in field_names):
        raise ValueError('field_names must all be valid identifiers')
    return field_names

print(parse_field_names("name, age, email"))
print(parse_field_names(["name", "age", "email"]))

---
## 2. Goose Typing and Abstract Base Classes

**Goose typing** = `isinstance(obj, AnABC)` -- checking against *abstract*
base classes rather than concrete types. ABCs formalize interfaces and
let you do safe runtime type checks while keeping Python flexible.

Key idea: "isinstance is fine... as long as the second argument is an ABC." -- Alex Martelli

In [ ]:
from collections import abc

# abc.Sized recognizes any class with __len__ -- no registration needed!
class Struggle:
    def __len__(self): return 23

s = Struggle()
print(f"isinstance(Struggle(), abc.Sized): {isinstance(s, abc.Sized)}")
print(f"issubclass(Struggle, abc.Sized):   {issubclass(Struggle, abc.Sized)}")

# Standard ABCs in collections.abc
for name in ['Iterable', 'Container', 'Sized', 'Sequence',
             'MutableSequence', 'Mapping', 'Set', 'Callable', 'Hashable']:
    print(f"  abc.{name:20s} exists: {hasattr(abc, name)}")

### Subclassing an ABC: FrenchDeck2

When you subclass an ABC, you **must** implement all abstract methods,
or `TypeError` is raised at instantiation. In return, you inherit
concrete mixin methods for free.

In [ ]:
import collections
from collections import abc as collections_abc

Card = collections.namedtuple('Card', ['rank', 'suit'])

class FrenchDeck2(collections_abc.MutableSequence):
    ranks = [str(n) for n in range(2, 11)] + list('JQKA')
    suits = 'spades diamonds clubs hearts'.split()

    def __init__(self):
        self._cards = [Card(rank, suit) for suit in self.suits
                       for rank in self.ranks]

    def __len__(self):
        return len(self._cards)

    def __getitem__(self, position):
        return self._cards[position]

    # Required by MutableSequence:
    def __setitem__(self, position, value):
        self._cards[position] = value

    def __delitem__(self, position):
        del self._cards[position]

    def insert(self, position, value):
        self._cards.insert(position, value)

deck2 = FrenchDeck2()
print(f"Cards: {len(deck2)}")

# Inherited concrete methods from MutableSequence:
random.shuffle(deck2)
print(f"Shuffled first 3: {deck2[:3]}")
print(f"Has .count():     {hasattr(deck2, 'count')}")
print(f"Has .index():     {hasattr(deck2, 'index')}")
print(f"Has .append():    {hasattr(deck2, 'append')}")
print(f"Has .reverse():   {hasattr(deck2, 'reverse')}")

### Defining a Custom ABC: Tombola

A custom ABC with **abstract** (`load`, `pick`) and **concrete**
(`loaded`, `inspect`) methods. Concrete methods are built entirely
on the abstract interface.

In [ ]:
import abc

class Tombola(abc.ABC):
    @abc.abstractmethod
    def load(self, iterable):
        """Add items from an iterable."""

    @abc.abstractmethod
    def pick(self):
        """Remove item at random, returning it.
        Should raise LookupError when empty.
        """

    def loaded(self):
        """Return True if there is at least one item."""
        return bool(self.inspect())

    def inspect(self):
        """Return a sorted tuple of current items."""
        items = []
        while True:
            try:
                items.append(self.pick())
            except LookupError:
                break
        self.load(items)
        return tuple(items)

# Try to instantiate without implementing abstract methods:
class Fake(Tombola):
    def pick(self): return 13

try:
    f = Fake()
except TypeError as e:
    print(f"Cannot instantiate: {e}")

#### Concrete subclasses of Tombola

In [ ]:
import random

class BingoCage(Tombola):
    def __init__(self, items):
        self._randomizer = random.SystemRandom()
        self._items = []
        self.load(items)

    def load(self, items):
        self._items.extend(items)
        self._randomizer.shuffle(self._items)

    def pick(self):
        try:
            return self._items.pop()
        except IndexError:
            raise LookupError('pick from empty BingoCage')

class LottoBlower(Tombola):
    def __init__(self, iterable):
        self._balls = list(iterable)  # Defensive: copy + fail fast

    def load(self, iterable):
        self._balls.extend(iterable)

    def pick(self):
        try:
            position = random.randrange(len(self._balls))
        except ValueError:
            raise LookupError('pick from empty LottoBlower')
        return self._balls.pop(position)

    def loaded(self):       # Override: much cheaper
        return bool(self._balls)

    def inspect(self):      # Override: one-liner
        return tuple(self._balls)

# Test both
cage = BingoCage(range(1, 10))
print(f"BingoCage pick: {cage.pick()}, loaded: {cage.loaded()}")

blower = LottoBlower(range(1, 46))
print(f"LottoBlower pick: {blower.pick()}, items left: {len(blower.inspect())}")

# Both pass isinstance checks
print(f"isinstance(cage, Tombola):   {isinstance(cage, Tombola)}")
print(f"isinstance(blower, Tombola): {isinstance(blower, Tombola)}")

---
## 3. Virtual Subclasses and register

`ABC.register()` declares a class as a **virtual subclass** -- it passes
`isinstance`/`issubclass` checks but does **not** inherit any methods.
The class does not appear in `__mro__`.

In [ ]:
from random import randrange

@Tombola.register
class TomboList(list):
    """A list that is a virtual subclass of Tombola."""
    def pick(self):
        if self:
            position = randrange(len(self))
            return self.pop(position)
        else:
            raise LookupError('pop from empty TomboList')

    load = list.extend

    def loaded(self):
        return bool(self)

    def inspect(self):
        return tuple(self)

t = TomboList(range(20))
print(f"TomboList pick: {t.pick()}")

# Virtual subclass checks
print(f"issubclass(TomboList, Tombola): {issubclass(TomboList, Tombola)}")
print(f"isinstance(t, Tombola):         {isinstance(t, Tombola)}")

# But Tombola is NOT in __mro__
print(f"TomboList.__mro__: {TomboList.__mro__}")

### __subclasshook__: Structural Recognition by ABCs

Some ABCs (like `Sized`) implement `__subclasshook__` to recognize
classes **structurally** -- without any registration or inheritance.

In [ ]:
import inspect
from collections import abc

# Look at Sized's __subclasshook__
print(inspect.getsource(abc.Sized.__subclasshook__))

# Any class with __len__ is recognized as Sized
class MyThing:
    def __len__(self):
        return 42

print(f"isinstance(MyThing(), abc.Sized): {isinstance(MyThing(), abc.Sized)}")
print(f"issubclass(MyThing, abc.Sized):   {issubclass(MyThing, abc.Sized)}")

# But Sequence does NOT have __subclasshook__ -- too risky
# (mappings also have __len__ + __getitem__ + __iter__)
print(f"issubclass(MyThing, abc.Sequence): {issubclass(MyThing, abc.Sequence)}")

---
## 4. Static Protocols with typing.Protocol

`typing.Protocol` (PEP 544) brings **static duck typing** to Python.
Define a Protocol subclass; any class implementing the required methods
is *consistent-with* that protocol -- no inheritance needed.

In [ ]:
from typing import TypeVar, Protocol, runtime_checkable, Any

T = TypeVar('T')

class Repeatable(Protocol):
    """Any type that supports multiplication by int."""
    def __mul__(self: T, repeat_count: int) -> T: ...

# double() works with anything Repeatable -- static duck typing
RT = TypeVar('RT', bound=Repeatable)

def double(x: RT) -> RT:
    return x * 2

# Works with str, list, int, float, Fraction...
print(double(1.5))          # 3.0
print(double('A'))           # AA
print(double([10, 20, 30]))  # [10, 20, 30, 10, 20, 30]

from fractions import Fraction
print(double(Fraction(2, 5)))  # 4/5

### @runtime_checkable: Using Protocols with isinstance

Decorating a Protocol with `@runtime_checkable` allows `isinstance`
checks at runtime. But beware: runtime checks only verify method
**presence**, not signatures or return types.

In [ ]:
from typing import SupportsFloat, SupportsComplex

# Runtime checks
print(f"isinstance(1.0, SupportsFloat):   {isinstance(1.0, SupportsFloat)}")
print(f"isinstance(42, SupportsFloat):     {isinstance(42, SupportsFloat)}")
print(f"isinstance('x', SupportsFloat):    {isinstance('x', SupportsFloat)}")

### Defining a Custom Static Protocol: RandomPicker

In [ ]:
from typing import Protocol, runtime_checkable, Any
import random

@runtime_checkable
class RandomPicker(Protocol):
    def pick(self) -> Any: ...

class SimplePicker:
    """Implements RandomPicker protocol without inheriting from it."""
    def __init__(self, items):
        self._items = list(items)
        random.shuffle(self._items)

    def pick(self):
        return self._items.pop()

popper = SimplePicker([1, 2, 3, 4, 5])

# Static duck typing: SimplePicker is consistent-with RandomPicker
print(f"isinstance check:   {isinstance(popper, RandomPicker)}")
print(f"Picked item:        {popper.pick()}")

# Extending a protocol
@runtime_checkable
class LoadableRandomPicker(RandomPicker, Protocol):
    """Extended protocol -- must also have load()."""
    def load(self, iterable) -> None: ...

# SimplePicker does NOT satisfy LoadableRandomPicker (no load method)
print(f"SimplePicker is LoadableRandomPicker? {isinstance(popper, LoadableRandomPicker)}")

# But BingoCage (defined earlier) does have both pick() and load()
print(f"BingoCage is RandomPicker?           {isinstance(cage, RandomPicker)}")
print(f"BingoCage is LoadableRandomPicker?   {isinstance(cage, LoadableRandomPicker)}")

---
## 5. Protocol Design Best Practices

| Guideline | Rationale |
|---|---|
| **Narrow protocols** (1-2 methods) | More reusable, composable |
| **Define near client code** | Easy to extend, test with mocks |
| **Naming: SupportsX, HasX, Verb+er** | Clear intent |
| **Extend by deriving** | Don't break existing consumers |
| **Prefer duck typing at runtime** | `try/except` > `isinstance` for most cases |
| **Use ABCs for frameworks** | When you need enforced contracts |
| **Use Protocols for libraries** | When callers should stay decoupled |

---
## 6. Summary: When to Use What

| Need | Approach | Tool |
|---|---|---|
| Maximum flexibility, small codebase | Duck typing | Just implement methods |
| Runtime enforcement of interface contracts | Goose typing | `abc.ABC` + `isinstance` |
| Static analysis, IDE support | Static protocols | `typing.Protocol` |
| Framework extension points | Custom ABCs | `abc.ABC` + `@abstractmethod` |
| Registering third-party classes | Virtual subclasses | `ABC.register()` |
| Both static + runtime checks | Runtime-checkable protocol | `@runtime_checkable` |

---
## Knowledge Check

In [ ]:
# Q1: What happens if you subclass an ABC but forget an abstract method?
class IncompleteSequence(collections_abc.Sequence):
    def __getitem__(self, index):
        return index

# Try to instantiate:
try:
    seq = IncompleteSequence()
except TypeError as e:
    print(f"Q1 Answer: {e}")
    # You get TypeError at instantiation time (not at class definition time!)

In [ ]:
# Q2: Does a virtual subclass inherit methods from the ABC?
@Tombola.register
class EmptyRegister(list):
    pass

print(f"Q2: issubclass? {issubclass(EmptyRegister, Tombola)}")
print(f"    Has .pick from Tombola? {'pick' in dir(EmptyRegister) and callable(getattr(EmptyRegister, 'pick', None))}")
print(f"    __mro__: {EmptyRegister.__mro__}")
# Answer: No! Virtual subclasses inherit NOTHING from the ABC.

In [ ]:
# Q3: Can a Protocol be extended?
@runtime_checkable
class Drawable(Protocol):
    def draw(self) -> None: ...

@runtime_checkable
class DrawableWithColor(Drawable, Protocol):
    def set_color(self, color: str) -> None: ...

class Circle:
    def draw(self) -> None:
        print("Drawing circle")

class ColorCircle:
    def draw(self) -> None:
        print("Drawing circle")
    def set_color(self, color: str) -> None:
        print(f"Setting color to {color}")

c = Circle()
cc = ColorCircle()
print(f"Q3: Circle is Drawable?              {isinstance(c, Drawable)}")
print(f"    Circle is DrawableWithColor?      {isinstance(c, DrawableWithColor)}")
print(f"    ColorCircle is Drawable?          {isinstance(cc, Drawable)}")
print(f"    ColorCircle is DrawableWithColor? {isinstance(cc, DrawableWithColor)}")

---
*End of Chapter 13 notebook. Key takeaway: Python's four typing approaches
are complementary -- use the right tool for each situation.*